# Simulation transect quick look

Find NetCDF simulation outputs, combine their consecutive time steps into one lazy xarray dataset, inspect its structure, and make a quick variable plot.

So far (16/11/26) used with pangeo-meso env on Spirit.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
# sys.path.append('../tools/')
sys.path.append('tools/')
from imports import *
from info import *
from tools_generic import *
from mapping import *

# Loading data

In [ ]:
# The notebook may be launched from either awaca_postdoc/ or notebooks/.
sim_dir = next(
    (path.resolve() for path in (Path("../sim_outputs"), Path("../../sim_outputs")) if path.is_dir()),
    None,
)

if sim_dir is None:
    raise FileNotFoundError(f"Could not find sim_outputs from {Path.cwd()}")

print(f"Simulation directory: {sim_dir}")

In [ ]:
netcdf_suffixes = {".nc", ".nc4", ".cdf", ".netcdf"}
sim_files = sorted(
    path for path in sim_dir.rglob("*") if path.is_file() and path.suffix.lower() in netcdf_suffixes
)

print(f"Found {len(sim_files)} NetCDF files")
for path in sim_files[:20]:
    print(path.relative_to(sim_dir))
if len(sim_files) > 20:
    print(f"... and {len(sim_files) - 20} more")

if not sim_files:
    raise FileNotFoundError(f"No NetCDF files found under {sim_dir}")

In [ ]:
# Open the daily files as one lazy dataset along their shared time coordinate.
ds = xr.open_mfdataset(
    sim_files,
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    chunks={"time_counter": 24},
)

print(f"Combined {len(sim_files)} files")
print(f"Time range: {ds.time_counter.min().values} to {ds.time_counter.max().values}")
display(ds)

# Analysis

## Mapping

In [ ]:
# Change these three values to customize the map.
map_variable = "sicf"
map_vmin = 0
map_vmax = 0.01
# map_vmin, map_vmax = None,None

# fig, ax, mesh = 
plot_antarctic_map(
    ds,
    variable=map_variable,
    vmin=map_vmin,
    vmax=map_vmax,
)

## Time series and diurnal cycles

The functions below return `(fig, ax, line)`. Domain means use cosine-latitude weights; pass a boolean `xarray.DataArray` as `mask` to limit the spatial average. Grid-cell functions choose the closest model point to the requested longitude/latitude. For variables with vertical levels, change `level_index` (default: first level).

In [ ]:
# Edit these settings, then choose full-domain or masked-domain plots.
timeseries_variable = "snowerosion"
subdomain_mask = None  # Boolean DataArray on lat/lon, or None for the full domain.
timeseries_level_index = 0

In [ ]:
plot_domain_mean_timeseries(
    ds,
    timeseries_variable,
    mask=subdomain_mask,
    level_index=timeseries_level_index,
)
plot_domain_mean_diurnal_cycle(
    ds,
    timeseries_variable,
    mask=subdomain_mask,
    level_index=timeseries_level_index,
)


In [ ]:
timeseries_variable = "SWdnSFC"
# These functions select the closest simulation grid cell to this location.
site="D17"
point_lon = sites_coord[site][0]
point_lat = sites_coord[site][1]

plot_gridcell_timeseries(
    ds,
    timeseries_variable,
    lon=point_lon,
    lat=point_lat,
    level_index=timeseries_level_index,
)
plot_gridcell_mean_diurnal_cycle(
    ds,
    timeseries_variable,
    lon=point_lon,
    lat=point_lat,
    level_index=timeseries_level_index,
)

In [ ]:
timeseries_variable = "snowerosion"
sites_list = ["DDU", "D17", "D47", "D85", "DMC"]  # Add your list of sites here

# Create a figure and axis explicitly to hold all lines
fig, ax = plt.subplots(figsize=(10, 4))

for site in sites_list:
  point_lon = sites_coord[site][0]
  point_lat = sites_coord[site][1]

  plot_gridcell_timeseries(
      ds,
      timeseries_variable,
      lon=point_lon,
      lat=point_lat,
      level_index=timeseries_level_index,
      title=f"{timeseries_variable}",
      ax=ax,  # Pass the shared axis if the function supports it
  )

# plt.legend(sites_list, title="Sites",bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()

In [ ]:
# timeseries_variable = "t2m"
# sites_list = ["DDU", "D17", "D47", "D85", "DMC"]  # Add your list of sites here
ymin=0
ymin,ymax=None,None

# Create a figure and axis explicitly to hold all lines
fig, ax = plt.subplots(figsize=(5, 4))

for site in sites_list:
  point_lon = sites_coord[site][0]
  point_lat = sites_coord[site][1]

  plot_gridcell_mean_diurnal_cycle(
        ds,
        timeseries_variable,
        lon=point_lon,
        lat=point_lat,
        level_index=timeseries_level_index,
        ax=ax,  # Pass the shared axis if the function supports it
        title=f"{timeseries_variable}",
        ymin=ymin
    )

# plt.legend(sites_list, title="Sites",bbox_to_anchor=(1.05, 1), loc="upper left")
plt.show()